# Thesis OOD Baseline Setup

This notebook documents the workflow for the **out-of-distribution baseline** on xBD/xView2 using a building-level damage classification pipeline.

The OOD split used in this notebook is based on the results of:

`out-of-distribution_preprocessing.ipynb`

---

## Core Principle

- Preprocessing and model training run directly in Python
- Notebook is used for:
  - Data inspection
  - Sanity checks
  - Training control
  - Evaluation
  - Visualization

The official xView2 Docker pipeline is not used as the main experimental setup, because the thesis isolates the damage classification task from localization by using ground truth building polygons.

---

## Out-of-Distribution Concept

In this baseline, **out-of-distribution** means that the geographical disaster locations used for testing and final holdout evaluation do **not appear in training**.

The split is therefore based on disaster locations, not random images or random buildings.

This avoids location leakage and makes evaluation stricter, because the model is tested on locations it has never seen during training.

---

## Required Data

The original xBD/xView2 folders must be placed on Desktop:

- `train`
- `test`
- `hold`

These contain:

- images
- labels

The OOD preprocessing notebook creates the new folders:

- `OOD_train`
- `OOD_test`
- `OOD_hold`

and the processed crop folder:

- `OOD_processed`

---

## Main Experimental Design

The thesis does not use the original end-to-end competition pipeline as the primary method.

Instead, the pipeline is:

1. Read ground truth polygons from label JSON files
2. Build a building-level metadata table
3. Create an OOD split where disaster locations do not overlap
4. Extract pre/post building crops
5. Stack pre and post crops into 6-channel tensors
6. Train a ResNet50-based classifier on damage labels
7. Evaluate on OOD test and OOD hold splits

This removes localization as a confounding factor and isolates the damage classification task under an out-of-distribution evaluation setting.

---

## Required External Apps

- `miniconda` or another Python environment manager

---

## Create Environment

```bash
conda create -n thesis_xview python=3.10
conda activate thesis_xview

In [1]:
import sys
!{sys.executable} -m pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import torch
import numpy as np
import pandas as pd
import cv2
import shapely

print("All packages loaded successfully")
print("Torch version:", torch.__version__)

All packages loaded successfully
Torch version: 2.8.0


# Training, Validation, and Class Weighting Strategy

## 1. Data Splits: Train, Test, and Hold

The dataset is divided into three distinct subsets, each with a specific role in the machine learning pipeline:

### Train Split
The **training set** is used to learn the model parameters. During this phase, the model updates its weights based on the input data and corresponding labels.

### Test Split
The **test set** is used during training to evaluate model performance after each epoch. It serves as a **validation set** for:

- monitoring learning progress  
- selecting the best model (e.g., best epoch)  
- comparing configurations or hyperparameters  

Importantly, the model does **not learn from the test set**. It is only used for evaluation.

### Hold Split
The **holdout set** is used for the **final evaluation** after training is complete. It provides an unbiased estimate of model performance because:

- it is not used during training  
- it is not used for model selection  

This separation ensures that reported results reflect true generalization.

---

### Summary of Roles

| Split | Purpose |
|------|--------|
| Train | Learn model parameters |
| Test | Model selection and monitoring |
| Hold | Final unbiased evaluation |

---

## 2. Class Imbalance and Class Weights

### Problem: Imbalanced Data

In the dataset, the distribution of damage classes is highly imbalanced. For example:

- "no-damage" appears far more frequently  
- "minor", "major", and "destroyed" are much less common  

Without correction, the model would tend to predict the majority class ("no-damage") to minimize loss, leading to poor performance on rare but important damage categories.

---

### Solution: Class-Weighted Loss

To address this, a **class-weighted cross-entropy loss** is used.

Each class is assigned a weight:

```text
[0.3564, 3.0456, 2.0528, 2.6418]

In [ ]:
import sys
!{sys.executable} model/OOD_classification_baseline.py

Loading OOD data...

Split sizes:
split
OOD_train    160276
OOD_test      54329
OOD_hold      52175
Name: count, dtype: int64

Train label distribution:
damage_label
no-damage       112434
major-damage     19519
destroyed        15167
minor-damage     13156
Name: count, dtype: int64

Test label distribution:
damage_label
no-damage       32574
minor-damage    12713
destroyed        6022
major-damage     3020
Name: count, dtype: int64

Hold label distribution:
damage_label
no-damage       51815
minor-damage      247
major-damage       77
destroyed          36
Name: count, dtype: int64

Image overlap check:
OOD_train ∩ OOD_test: 0
OOD_train ∩ OOD_hold: 0
OOD_test ∩ OOD_hold: 0

Location overlap check:
OOD_train ∩ OOD_test: 0
OOD_train ∩ OOD_hold: 0
OOD_test ∩ OOD_hold: 0

PASS: no image or location overlap.

Using device: mps

Using class weights:
[0.35637796 3.0456827  2.0528204  2.641854  ]

Starting OOD training...
Epoch 1/10:  16%|██▍            | 820/5009 [03:28<19:00,  3.67it/s, los